# Machine Learning PAM50 Classification of Multi-Omics Breast Cancer Data

## Data Preprocessing - METABRIC

In [37]:
import pandas as pd
from collections import Counter

# Base folder where all cBioPortal .txt files are stored
base = "metabrics-data/"

# -----------------------------
# Clinical sample data
# Contains PAM50 molecular subtype labels
# -----------------------------
clinical = pd.read_csv(
    base + "data_clinical_patient.txt",
    sep="\t",
    comment="#"
)

# -----------------------------
# mRNA expression data (z-score normalized)
# Rows = genes, columns = samples
# Used to extract PAM50 gene expression features
# -----------------------------
expr = pd.read_csv(
    base + "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt",
    sep="\t"
)

# -----------------------------
# Copy Number Alteration (CNA) data
# Values typically: -2, -1, 0, 1, 2
# Used for ERBB2, TP53, MYC, etc.
# -----------------------------
cna = pd.read_csv(
    base + "data_cna.txt",
    sep="\t",
    comment="#"
)

# -----------------------------
# Somatic mutation calls
# Long-format table: one row per mutation event
# Will later be converted to a binary mutation matrix
# -----------------------------
mut = pd.read_csv(
    base + "data_mutations.txt",
    sep="\t",
    comment="#"
)

In [38]:
# Extract PAM50 labels

# Extract PAM50 labels from patient-level clinical data, drop missing
pam50 = clinical[["PATIENT_ID", "CLAUDIN_SUBTYPE"]].dropna()

# Rename column for convenience
pam50 = pam50.rename(columns={"CLAUDIN_SUBTYPE": "PAM50"})

# Rename PATIENT_ID to Sample_ID for merging later - these are all equal according to data_clinical_sample.txt
pam50 = pam50.rename(columns={"PATIENT_ID": "Sample_ID"})

# Print counts of each PAM50 classification
print(Counter(pam50["PAM50"]))

pam50.head()

Counter({'LumA': 700, 'LumB': 475, 'Her2': 224, 'claudin-low': 218, 'Basal': 209, 'Normal': 148, 'NC': 6})


,Sample_ID,PAM50
0,MB-0000,claudin-low
1,MB-0002,LumA
2,MB-0005,LumB
3,MB-0006,LumB
4,MB-0008,LumB


In [39]:
# --- Define PAM50 genes ---
pam50_genes = [
    "ACTR3B","ANLN","BAG1","BCL2","BLVRA","CCNB1","CEP55","CXXC5","DKK1","EGFR",
    "ESR1","EXO1","FOXC1","FOXB1","GPR160","GRB7","KIF2C","KRT17","KRT5","MAPT",
    "MELK","MIA","MKI67","MLPH","NDC80","NKI1","NUF2","ORC6","PGR","PHGDH","PTTG1",
    "RCHY1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T","CDCA1","KIF4A",
    "KIF14","CENPF","CDC20","MMP11","MMP9","CCNE1","CCNE2","FGFR4"
]

# --- Filter expression matrix for PAM50 genes ---
pam50_expr = expr[expr['Hugo_Symbol'].isin(pam50_genes)].copy()

# --- Set gene symbols as index ---
pam50_expr.set_index('Hugo_Symbol', inplace=True)

# --- Keep only sample columns (exclude Entrez ID or other metadata) ---
sample_cols = [col for col in pam50_expr.columns if col.startswith('MB-')]
pam50_expr = pam50_expr[sample_cols]

# Transpose
pam50_expr = pam50_expr.T

# Rename gene columns to have '_expr' suffix
pam50_expr.columns = [f"{col}_expr" for col in pam50_expr.columns]

# Adjust index
pam50_expr.index.name = 'Sample_ID'  # name the index
pam50_expr.reset_index(inplace=True)

pam50_expr.head()

,Sample_ID,MKI67_expr,FGFR4_expr,MELK_expr,PHGDH_expr,RCHY1_expr,TMEM45B_expr,RRM2_expr,CXXC5_expr,NDC80_expr,...,KRT5_expr,GRB7_expr,TYMS_expr,PGR_expr,ANLN_expr,SFRP1_expr,MIA_expr,GPR160_expr,CEP55_expr,SLC39A6_expr
0,MB-0362,-0.3049,-0.6509,0.5591,0.8807,0.0095,-1.0768,1.3946,0.4153,-1.0091,...,-0.2661,-0.2220,1.2945,-0.1109,-0.8559,0.0240,-0.2503,0.1582,-0.4900,0.8541
1,MB-0346,0.3986,4.5695,0.8176,1.1893,1.4690,1.7159,3.7655,0.5993,-0.2614,...,-0.8248,3.1187,1.9027,-0.9229,0.0370,-0.6845,-0.5261,0.0866,0.8121,-1.0916
2,MB-0386,-0.4301,-0.7584,0.5910,-1.0947,-1.1718,0.0645,0.7738,-0.1557,-1.0134,...,-0.6812,-0.7700,-0.5616,1.1598,0.7810,0.6685,0.1072,-0.3504,-0.0626,0.6336
3,MB-0574,-0.0579,-0.2641,-0.4552,-0.2619,0.3176,-0.3569,0.0832,0.3017,-0.8109,...,-0.9799,-0.2478,0.1746,1.0582,-0.8023,-0.7551,-0.2246,0.3012,-0.3206,1.0778
4,MB-0185,0.2596,-0.5279,1.3351,0.6176,0.4223,0.1526,2.1131,0.7284,-0.5942,...,-1.0371,-0.2288,1.1307,1.3549,0.3652,-1.4018,-0.0075,-0.1648,1.7850,-0.5128


In [40]:
# Convert column names to strings and strip whitespace
cna.columns = cna.columns.str.strip()

# CNA genes of interest
cna_genes = ["ERBB2","MYC","CCND1","FGFR1","PTEN","RB1","TP53","PIK3CA"]

# Set Hugo_Symbol as index
cna = cna.set_index("Hugo_Symbol")

# Filter only the genes we care about
cna = cna.loc[cna_genes]

# Drop Entrez_Gene_Id column
cna = cna.drop(columns=["Entrez_Gene_Id"])

# Transpose so samples are rows
cna = cna.T

# Rename columns with _cna suffix
cna.columns = [gene + "_cna" for gene in cna.columns]

# Add SAMPLE_ID column
cna["SAMPLE_ID"] = cna.index

# Adjust index
cna.index.name = 'Sample_ID'  # name the index
cna.reset_index(inplace=True)

# Remove SAMPLE_ID
cna = cna.drop(columns=["SAMPLE_ID"])

cna.head()

,Sample_ID,ERBB2_cna,MYC_cna,CCND1_cna,FGFR1_cna,PTEN_cna,RB1_cna,TP53_cna,PIK3CA_cna
0,MB-0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,MB-0039,0.0,0.0,2.0,0.0,0.0,-1.0,0.0,0.0
2,MB-0045,0.0,0.0,0.0,-1.0,0.0,-1.0,-1.0,2.0
3,MB-0046,2.0,0.0,0.0,-1.0,0.0,-1.0,-1.0,0.0
4,MB-0048,2.0,1.0,1.0,0.0,1.0,0.0,-1.0,1.0


In [41]:
# List of genes to include
mut_genes = ["TP53","PIK3CA","GATA3","MAP3K1","CDH1","AKT1","PTEN","RB1","BRCA1","BRCA2"]

# Keep only the genes we want
mut = mut[mut["Hugo_Symbol"].isin(mut_genes)]

# Create long → wide binary mutation matrix
mut_matrix = (
    mut.groupby(["Tumor_Sample_Barcode", "Hugo_Symbol"])
       .size()
       .unstack(fill_value=0)
)

# Convert counts to 0/1
mut_matrix = (mut_matrix > 0).astype(int)

# Rename columns for clarity
mut_matrix.columns = [g + "_mut" for g in mut_matrix.columns]

# Add SAMPLE_ID column for merging
mut_matrix["SAMPLE_ID"] = mut_matrix.index

# Adjust index
mut_matrix.index.name = 'Sample_ID'  # name the index
mut_matrix.reset_index(inplace=True)

# Remove SAMPLE_ID
mut_matrix = mut_matrix.drop(columns=["SAMPLE_ID"])

mut_matrix.head()

,Sample_ID,AKT1_mut,BRCA1_mut,BRCA2_mut,CDH1_mut,GATA3_mut,MAP3K1_mut,PIK3CA_mut,PTEN_mut,RB1_mut,TP53_mut
0,MB-0002,0,0,0,0,0,0,0,0,0,1
1,MB-0005,0,0,0,0,0,0,1,0,0,0
2,MB-0006,0,0,0,0,0,0,1,0,0,0
3,MB-0008,0,0,0,0,0,0,0,0,0,1
4,MB-0010,0,0,0,0,1,0,1,0,0,1


In [42]:
# Merge

df = pam50 \
    .merge(pam50_expr, on="Sample_ID") \
    .merge(cna, on="Sample_ID") \
    .merge(mut_matrix, on="Sample_ID", how="left")

df = df.fillna(0)

In [43]:
# Change Sample_ID to index
df.set_index('Sample_ID', inplace=True)

# Add PAM50_Label column based on PAM50 word labels
df['PAM50_Label'] = df['PAM50']

# Convert PAM50 column with word labels to integer label column
pam50_map = {'LumA': 0, 'LumB': 1, 'Her2': 2, 'Basal': 3, 'Normal': 4, 'claudin-low': 5}
df['PAM50'] = df['PAM50'].map(pam50_map)

# Drop "NC" rows from METABRIC data
df = df[df['PAM50_Label'] != 'NC']

# Save with index column
df.to_csv("metabric_data.csv", index=True)

print(df.shape)
df.head()

(1974, 67)


,PAM50,MKI67_expr,FGFR4_expr,MELK_expr,PHGDH_expr,RCHY1_expr,TMEM45B_expr,RRM2_expr,CXXC5_expr,NDC80_expr,...,BRCA1_mut,BRCA2_mut,CDH1_mut,GATA3_mut,MAP3K1_mut,PIK3CA_mut,PTEN_mut,RB1_mut,TP53_mut,PAM50_Label
Sample_ID,,,,,,,,,,,,,,,,,,,,,
MB-0000,5.0,-1.0974,-0.8030,-2.1862,0.7686,-0.9575,-1.0049,-1.2647,-1.2763,-1.1544,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,claudin-low
MB-0002,0.0,-1.2240,-0.5858,-0.6482,-0.8107,1.9562,-0.1271,-0.7466,-0.1827,-0.8541,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,LumA
MB-0005,1.0,-1.2915,-0.6090,0.4141,0.0154,0.1824,0.0402,-0.2317,-1.6266,1.5478,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,LumB
MB-0006,1.0,-1.1148,1.4439,-0.4960,-2.1686,0.3362,0.2135,-0.1262,0.9086,0.5335,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,LumB
MB-0008,1.0,-0.7003,-0.9569,0.3547,0.5362,0.2710,-0.7815,1.8113,-0.4658,0.1975,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,LumB
